# start 

In [1]:
# -----######-----###### RENAME FINAL: FULL CONTROL + PUkw CORRECTED -----######-----######
import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime

def _rename_2304_kwtagging_GET_renamed_files_from_txt(
    txt_path,
    custom_artist="DJ_Selphi",
    custom_genre="Salsa",
    custom_label="Bachata",
    custom_release_date="2017_01_01",
    custom_purchase_date=""
):
    """
    Rename files using a kw-tag structure with optional custom overrides.
    Pulls from TXT if any override is left blank.

    Args:
        txt_path (str): Path to UTF-16 tab-separated metadata file
        custom_artist (str): Optional override for artist (max 25 chars)
        custom_genre (str): Optional override for genre
        custom_label (str): Optional override for label
        custom_release_date (str): Optional override in YYYY_MM_DD format
        custom_purchase_date (str): Optional override in YYYY_MM_DD format

    Returns:
        pd.DataFrame: Original DataFrame + 'Renamed_Path' column
    """
    tqdm.pandas()
    df = pd.read_csv(txt_path, sep="\t", encoding='utf-16', engine='python')
    df.columns = df.columns.str.strip()

    def clean(s):
        return (
            str(s)
            .replace(" ", "_").replace("/", "___").replace(",", "_")
            .replace("(", "").replace(")", "").replace("!", "")
            .replace("&", "and").replace("’", "").replace("'", "")
            .replace("¿", "").replace("¡", "").replace(":", "")
            .replace(";", "").strip()
        )

    def extract_mix(title):
        title_lower = title.lower()
        if "remix" in title_lower or "mix" in title_lower:
            return clean(title)
        return "original"

    def format_filename(row):
        title = clean(row.get('Track Title', ''))[:25]
        remix = extract_mix(row.get('Track Title', ''))

        artist_val = clean(custom_artist)[:25] if custom_artist else clean(row.get('Artist', ''))[:25]
        genre_val = clean(custom_genre) if custom_genre else clean(row.get('Genre', ''))
        label_val = clean(custom_label) if custom_label else clean(row.get('Label', ''))
        release_date_val = custom_release_date if custom_release_date else "2017_01_01"

        key = clean(row.get('Key', 'NA')) if pd.notna(row.get('Key', '')) else 'NA'
        bpm = str(int(round(float(row.get('BPM', 0))))) if pd.notna(row.get('BPM', 0)) else 'NA'

        date_added = pd.to_datetime(row.get('Date Added', datetime.today()), errors='coerce').strftime('%Y_%m_%d')
        purchase_date_val = (
            custom_purchase_date if custom_purchase_date
            else pd.to_datetime(row.get('Purchased', datetime.today()), errors='coerce').strftime('%Y_%m_%d')
        )

        ext = os.path.splitext(row.get('Location', ''))[1]

        new_name = (
            f"TRkw_{title}_ARkw_{artist_val}_MXkw_{remix}_KYkw_{key}_"
            f"BPkw_{bpm}_GNkw_{genre_val}_LBkw_{label_val}_RYkw_{release_date_val}_"
            f"PYkw_{purchase_date_val}{ext}"
        )

        if len(new_name) > 240:
            new_name = new_name[:230] + ext

        return new_name

    new_paths = []
    log_long_names = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Renaming files"):
        original_path = str(row.get('Location', '')).strip()

        if not os.path.isfile(original_path):
            print(f"❌ File not found (Row {i}): {original_path}")
            new_paths.append(None)
            continue

        new_filename = format_filename(row)
        new_path = os.path.join(os.path.dirname(original_path), new_filename)

        try:
            os.rename(original_path, new_path)
            new_paths.append(new_path)
        except Exception as e:
            print(f"❌ Error renaming (Row {i}): {e}")
            new_paths.append(None)
            log_long_names.append({
                "Index": i,
                "OriginalPath": original_path,
                "IntendedFilename": new_filename,
                "Error": str(e)
            })

    df['Renamed_Path'] = new_paths

    if log_long_names:
        pd.DataFrame(log_long_names).to_csv("long_name_errors_log.csv", index=False)
        print("📁 Saved log of failed renames to: long_name_errors_log.csv")

    return df


In [2]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

txt_path = "/Users/yerik/Downloads/salsa_new.txt"

df = _rename_2304_kwtagging_GET_renamed_files_from_txt(
    txt_path,
    custom_artist="Salsa Son Timba",           # ✅ or "" for TXT
    custom_genre="Salsa",                # ✅ or "" for TXT
    custom_label="Salsa_Venezuela",      # ✅ or "" for TXT
    custom_release_date="2013_01_01",    # ✅ or "" for fallback
    custom_purchase_date="2025_04_19"              # ✅ "" uses 'Purchased' column
)


Renaming files: 100%|███████████████████████████████████████████████████████| 16/16 [00:00<00:00, 3213.56it/s]


# check

In [ ]:
# check df
df = pd.read_csv(txt_path, sep="\t", encoding='latin1', engine='python')
df.columns = df.columns.str.strip()  # Clean up whitespaces, just in case

print("\n🔎 COLUMN NAMES:")
for i, col in enumerate(df.columns):
    print(f"{i:>2}: '{col}'")
df